# Exploratory Data Analysis and Data Preprocessing

## Objective
Understand the structure, content, and quality of the CFPB complaint data and prepare it for the RAG pipeline.

## Tasks
1. Load the full CFPB complaint dataset
2. Perform initial EDA to understand the data
3. Filter the dataset for specified products
4. Clean text narratives to improve embedding quality
5. Save the cleaned and filtered dataset


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 100)

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


## 1. Load the Dataset


In [ ]:
# Define file paths
data_dir = Path('../data')
raw_data_path = data_dir / 'raw' / 'complaints.csv'
output_data_path = data_dir / 'filtered_complaints.csv'

print(f"Loading data from: {raw_data_path}")
print("This may take a few minutes for large datasets...")

# Load the dataset (using low_memory=False to avoid mixed type warnings)
df = pd.read_csv(raw_data_path, low_memory=False)

print(f"\nDataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")


## 2. Initial Exploratory Data Analysis


In [ ]:
# Display basic information about the dataset
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"\nTotal number of complaints: {len(df):,}")
print(f"Number of columns: {len(df.columns)}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

print("\n" + "=" * 80)
print("DATA TYPES AND MISSING VALUES")
print("=" * 80)
print(df.info())


In [ ]:
# Check for missing values in key columns
print("=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)
key_columns = ['Product', 'Consumer complaint narrative']
missing_counts = df[key_columns].isnull().sum()
missing_pct = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage': missing_pct
})
print(missing_df)

# Count empty strings in Consumer complaint narrative
empty_narratives = (df['Consumer complaint narrative'].fillna('').str.strip() == '').sum()
print(f"\nEmpty or whitespace-only narratives: {empty_narratives:,} ({(empty_narratives/len(df)*100):.2f}%)")
print(f"Non-empty narratives: {len(df) - empty_narratives:,} ({((len(df) - empty_narratives)/len(df)*100):.2f}%)")


In [ ]:
# Analyze distribution of complaints across different Products
print("=" * 80)
print("PRODUCT DISTRIBUTION ANALYSIS")
print("=" * 80)
product_counts = df['Product'].value_counts()
print(f"\nTotal number of unique products: {df['Product'].nunique()}")
print(f"\nTop 20 products by complaint count:")
print(product_counts.head(20))

# Visualize product distribution (top 15)
plt.figure(figsize=(14, 8))
top_products = product_counts.head(15)
plt.barh(range(len(top_products)), top_products.values)
plt.yticks(range(len(top_products)), top_products.index)
plt.xlabel('Number of Complaints', fontsize=12)
plt.ylabel('Product', fontsize=12)
plt.title('Distribution of Complaints Across Products (Top 15)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nTop product: {product_counts.index[0]} ({product_counts.iloc[0]:,} complaints)")
print(f"Percentage of total: {(product_counts.iloc[0]/len(df)*100):.2f}%")


In [ ]:
# Calculate word count for Consumer complaint narrative
print("=" * 80)
print("NARRATIVE LENGTH ANALYSIS (Word Count)")
print("=" * 80)

# Function to calculate word count (handles NaN values)
def calculate_word_count(text):
    if pd.isna(text) or str(text).strip() == '':
        return 0
    return len(str(text).split())

# Calculate word counts
df['word_count'] = df['Consumer complaint narrative'].apply(calculate_word_count)

# Statistics for non-empty narratives
non_empty_mask = df['word_count'] > 0
non_empty_word_counts = df.loc[non_empty_mask, 'word_count']

print(f"\nWord Count Statistics (all records):")
print(df['word_count'].describe())

print(f"\nWord Count Statistics (non-empty narratives only):")
print(non_empty_word_counts.describe())

print(f"\nVery short narratives (1-10 words): {(df['word_count'] <= 10).sum():,}")
print(f"Short narratives (11-50 words): {((df['word_count'] > 10) & (df['word_count'] <= 50)).sum():,}")
print(f"Medium narratives (51-200 words): {((df['word_count'] > 50) & (df['word_count'] <= 200)).sum():,}")
print(f"Long narratives (201-500 words): {((df['word_count'] > 200) & (df['word_count'] <= 500)).sum():,}")
print(f"Very long narratives (>500 words): {(df['word_count'] > 500).sum():,}")


In [ ]:
# Visualize word count distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram for all narratives (including zeros)
axes[0].hist(df['word_count'], bins=100, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Word Count', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Word Count Distribution (All Records)', fontsize=14, fontweight='bold')
axes[0].axvline(df['word_count'].mean(), color='red', linestyle='--', label=f'Mean: {df["word_count"].mean():.1f}')
axes[0].legend()

# Histogram for non-empty narratives only (zoomed in)
axes[1].hist(non_empty_word_counts, bins=100, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Word Count', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Word Count Distribution (Non-Empty Narratives Only)', fontsize=14, fontweight='bold')
axes[1].axvline(non_empty_word_counts.mean(), color='red', linestyle='--', label=f'Mean: {non_empty_word_counts.mean():.1f}')
axes[1].legend()
axes[1].set_xlim(0, 1000)  # Focus on 0-1000 range

plt.tight_layout()
plt.show()

# Box plot for better visualization of outliers
plt.figure(figsize=(10, 6))
plt.boxplot([non_empty_word_counts], vert=True, patch_artist=True,
            boxprops=dict(facecolor='lightblue', alpha=0.7))
plt.ylabel('Word Count', fontsize=12)
plt.title('Word Count Distribution - Box Plot (Non-Empty Narratives)', fontsize=14, fontweight='bold')
plt.xticks([1], ['Consumer Complaint Narratives'])
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Filter Dataset for Specified Products

Filter for:
- Credit card
- Personal loan
- Savings account
- Money transfers


In [ ]:
# Check available product names to find exact matches
print("=" * 80)
print("PRODUCT NAME MATCHING")
print("=" * 80)

target_products = ['Credit card', 'Personal loan', 'Savings account', 'Money transfers']
print(f"\nTarget products to filter:")
for p in target_products:
    print(f"  - {p}")

# Find product names that contain our target keywords
print("\nAvailable product names in dataset:")
all_products = df['Product'].unique()

# Try to match products (case-insensitive, partial matching)
matched_products = []
for target in target_products:
    # Create pattern for matching
    if target.lower() == 'credit card':
        pattern = 'credit card'
    elif target.lower() == 'personal loan':
        pattern = 'personal loan'
    elif target.lower() == 'savings account':
        pattern = 'savings'
    elif target.lower() == 'money transfers':
        pattern = 'money transfer'
    else:
        pattern = target.lower()
    
    matches = [p for p in all_products if pattern in str(p).lower()]
    if matches:
        matched_products.extend(matches)
        print(f"\n'{target}' matches:")
        for m in matches:
            count = (df['Product'] == m).sum()
            print(f"  - '{m}' ({count:,} complaints)")

print("\n" + "=" * 80)
print("SELECTED PRODUCTS FOR FILTERING")
print("=" * 80)

# Define the exact product names to use based on the matches
# We'll use the most common variations
selected_products = [
    'Credit card',
    'Credit card or prepaid card',
    'Payday loan, title loan, or personal loan',
    'Checking or savings account',
    'Money transfer, virtual currency, or money service',
    'Money transfers'
]

# Remove duplicates while preserving order
selected_products = list(dict.fromkeys([p for p in selected_products if p in all_products]))

print(f"\nSelected products for filtering:")
for p in selected_products:
    count = (df['Product'] == p).sum()
    print(f"  - '{p}' ({count:,} complaints)")

total_before_filter = len(df)


In [ ]:
# Filter for selected products
df_filtered = df[df['Product'].isin(selected_products)].copy()

print("=" * 80)
print("FILTERING RESULTS")
print("=" * 80)
print(f"\nTotal records before filtering: {total_before_filter:,}")
print(f"Total records after product filtering: {len(df_filtered):,}")
print(f"Records removed: {total_before_filter - len(df_filtered):,} ({(total_before_filter - len(df_filtered))/total_before_filter*100:.2f}%)")

print(f"\nProduct distribution in filtered dataset:")
filtered_product_counts = df_filtered['Product'].value_counts()
for product, count in filtered_product_counts.items():
    print(f"  - {product}: {count:,} ({(count/len(df_filtered)*100):.2f}%)")


In [ ]:
# Remove records with empty or whitespace-only narratives
print("=" * 80)
print("REMOVING EMPTY NARRATIVES")
print("=" * 80)

before_empty_removal = len(df_filtered)
empty_mask = (df_filtered['Consumer complaint narrative'].fillna('').str.strip() == '')
empty_count = empty_mask.sum()

print(f"\nRecords before removing empty narratives: {before_empty_removal:,}")
print(f"Records with empty narratives: {empty_count:,} ({(empty_count/before_empty_removal*100):.2f}%)")

# Keep only records with non-empty narratives
df_filtered = df_filtered[~empty_mask].copy()

print(f"Records after removing empty narratives: {len(df_filtered):,}")
print(f"Records removed: {empty_count:,}")

# Verify no empty narratives remain
remaining_empty = (df_filtered['Consumer complaint narrative'].fillna('').str.strip() == '').sum()
print(f"\nVerification: Remaining empty narratives: {remaining_empty}")


## 5. Text Cleaning and Preprocessing

Clean the text narratives to improve embedding quality:
- Lowercasing text
- Removing special characters
- Removing boilerplate text
- Normalizing whitespace


In [ ]:
# Define text cleaning function
def clean_text(text):
    """
    Clean and normalize text for better embedding quality.
    
    Args:
        text: Input text string
        
    Returns:
        Cleaned text string
    """
    if pd.isna(text):
        return ''
    
    text = str(text)
    
    # Step 1: Convert to lowercase
    text = text.lower()
    
    # Step 2: Remove common boilerplate phrases
    boilerplate_patterns = [
        r'i am writing to file a complaint',
        r'i am writing to complain',
        r'i would like to file a complaint',
        r'this is a complaint regarding',
        r'i am filing this complaint',
        r'dear sir/madam',
        r'to whom it may concern',
    ]
    
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # Step 3: Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Step 4: Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Step 5: Remove phone numbers (various formats)
    text = re.sub(r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b', '', text)
    text = re.sub(r'\b\d{3}[-.\s]?\d{4}\b', '', text)
    
    # Step 6: Remove excessive punctuation (keep periods, question marks, exclamation marks)
    # Replace multiple punctuation with single instance
    text = re.sub(r'[!]{2,}', '!', text)
    text = re.sub(r'[?]{2,}', '?', text)
    text = re.sub(r'[.]{2,}', '.', text)
    
    # Step 7: Normalize whitespace
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with single space
    text = text.strip()  # Remove leading/trailing whitespace
    
    return text

# Apply cleaning
print("=" * 80)
print("TEXT CLEANING")
print("=" * 80)
print("\nApplying text cleaning to narratives...")

# Store original narrative for comparison
df_filtered['narrative_original'] = df_filtered['Consumer complaint narrative'].copy()

# Apply cleaning function
df_filtered['narrative_cleaned'] = df_filtered['Consumer complaint narrative'].apply(clean_text)

print(f"Text cleaning completed for {len(df_filtered):,} records.")

# Show some examples
print("\n" + "=" * 80)
print("CLEANING EXAMPLES (First 5 records)")
print("=" * 80)
for idx in range(min(5, len(df_filtered))):
    print(f"\n--- Example {idx + 1} ---")
    print(f"Original (first 200 chars):")
    print(df_filtered.iloc[idx]['narrative_original'][:200])
    print(f"\nCleaned (first 200 chars):")
    print(df_filtered.iloc[idx]['narrative_cleaned'][:200])
    print("-" * 80)


In [ ]:
# Check for narratives that became empty after cleaning
empty_after_cleaning = (df_filtered['narrative_cleaned'].str.strip() == '').sum()
print(f"\nNarratives that became empty after cleaning: {empty_after_cleaning:,}")

if empty_after_cleaning > 0:
    print(f"Removing {empty_after_cleaning:,} records with empty narratives after cleaning...")
    df_filtered = df_filtered[df_filtered['narrative_cleaned'].str.strip() != ''].copy()
    print(f"Remaining records: {len(df_filtered):,}")

# Update word count for cleaned narratives
df_filtered['word_count_cleaned'] = df_filtered['narrative_cleaned'].apply(calculate_word_count)

print(f"\nWord count statistics for cleaned narratives:")
print(df_filtered['word_count_cleaned'].describe())


## 6. Prepare Final Dataset and Save


In [ ]:
# Prepare final dataset with relevant columns
# Keep the cleaned narrative as the main narrative field
df_final = df_filtered.copy()

# Replace the original narrative with cleaned version
df_final['Consumer complaint narrative'] = df_final['narrative_cleaned']

# Drop temporary columns
columns_to_drop = ['narrative_original', 'narrative_cleaned', 'word_count_cleaned']
df_final = df_final.drop(columns=[col for col in columns_to_drop if col in df_final.columns])

print("=" * 80)
print("FINAL DATASET SUMMARY")
print("=" * 80)
print(f"\nFinal dataset shape: {df_final.shape}")
print(f"Total records: {len(df_final):,}")
print(f"Total columns: {len(df_final.columns)}")

print(f"\nProduct distribution in final dataset:")
final_product_counts = df_final['Product'].value_counts()
for product, count in final_product_counts.items():
    print(f"  - {product}: {count:,} ({(count/len(df_final)*100):.2f}%)")

print(f"\nFinal dataset columns:")
for col in df_final.columns:
    print(f"  - {col}")


In [ ]:
# Save the filtered and cleaned dataset
print("=" * 80)
print("SAVING DATASET")
print("=" * 80)

# Ensure output directory exists
output_data_path.parent.mkdir(parents=True, exist_ok=True)

# Save to CSV
df_final.to_csv(output_data_path, index=False)

print(f"\nDataset saved successfully to: {output_data_path}")
print(f"File size: {output_data_path.stat().st_size / (1024*1024):.2f} MB")

# Verify the saved file
df_verify = pd.read_csv(output_data_path)
print(f"\nVerification: Loaded {len(df_verify):,} records from saved file")
print(f"Columns match: {list(df_verify.columns) == list(df_final.columns)}")


## 7. Summary of Key Findings

### Dataset Overview
- **Total records in raw dataset**: [To be filled after running]
- **Records after filtering**: [To be filled after running]
- **Final records with valid narratives**: [To be filled after running]

### Key Findings from EDA

1. **Product Distribution**: [Summary of product distribution findings]

2. **Narrative Length Analysis**: 
   - Average word count: [To be filled]
   - Very short narratives (<10 words): [To be filled]
   - Very long narratives (>500 words): [To be filled]

3. **Data Quality**:
   - Percentage of records with empty narratives: [To be filled]
   - Records removed during filtering: [To be filled]

4. **Text Cleaning Impact**:
   - Narratives cleaned and normalized
   - Boilerplate text removed
   - Special characters and URLs removed
   - Text normalized to lowercase for better embedding consistency

### Next Steps
The cleaned and filtered dataset is now ready for:
- Chunking and embedding
- Vector store creation
- RAG pipeline implementation
